# 01 Data overview

First look at the raw US Accidents data (all 7.7M rows): column types, missing values, and how Severity is spread overall, by year and by Source.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # so `src` imports from the notebooks folder
from src import config
from src.data import load_raw_cached

config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# plot style: one blue ramp for severity 1 (light) to 4 (dark)
SEVERITY_COLORS = ['#86b6ef', '#3987e5', '#1c5cab', '#0d366b']
BLUE = '#2a78d6'
INK = '#52514e'
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK,
    'xtick.color': INK, 'ytick.color': INK, 'text.color': '#1a1a19',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e1e0d9', 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'font.size': 10,
})

## Load

The first run reads the CSV (about a minute) and saves `data/processed/raw.parquet`; after that it loads in seconds.

In [ ]:
df = load_raw_cached()
print(f'{len(df):,} rows, {df.shape[1]} columns, '
      f'{df.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory')

## Column types

In [ ]:
df.dtypes.to_frame('dtype')

## Missing values

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0].round(2)
missing.to_frame('% missing')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 0.32 * len(missing) + 0.8))
ax.barh(missing.index[::-1], missing.values[::-1], color=BLUE, height=0.6)
for y, value in enumerate(missing.values[::-1]):
    label = f'{value:.1f}%' if value >= 0.05 else '<0.1%'
    ax.text(value + 0.5, y, label, va='center', fontsize=8.5, color=INK)
ax.set_xlabel('% of rows missing')
ax.set_title('Missing values by column', loc='left', fontweight='bold')
ax.grid(axis='y', visible=False)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'missing_pct.png', dpi=150)
plt.show()

## Severity overall

In [ ]:
counts = df[config.TARGET].value_counts().sort_index()
pd.DataFrame({'rows': counts, '%': (counts / counts.sum() * 100).round(2)})

## Severity by year and by Source

The data comes from several APIs, so the way severity is labelled might change between sources or years. If it does, that matters for the models and the report.

In [ ]:
# share of each severity within a group, as a 100% stacked bar
def plot_severity_share(table, title, filename):
    share = table.div(table.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots(figsize=(8, 4))
    bottom = pd.Series(0.0, index=share.index)
    for sev, color in zip(config.CLASSES, SEVERITY_COLORS):
        ax.bar(share.index.astype(str), share[sev], bottom=bottom, color=color,
               edgecolor='#fcfcfb', linewidth=1, label=f'Severity {sev}')
        bottom += share[sev]
    ax.set_ylabel('% of accidents')
    ax.set_ylim(0, 100)
    ax.set_title(title, loc='left', fontweight='bold')
    ax.grid(axis='x', visible=False)
    ax.legend(ncol=4, loc='upper left', bbox_to_anchor=(0, -0.12), frameon=False)
    fig.tight_layout()
    fig.savefig(config.FIGURES_DIR / filename, dpi=150)
    plt.show()
    return share.round(1)

In [ ]:
year = pd.to_datetime(df['Start_Time'], format='mixed', errors='coerce').dt.year
by_year = pd.crosstab(year, df[config.TARGET])
by_year.assign(total=by_year.sum(axis=1))

In [ ]:
plot_severity_share(by_year, 'Severity mix by year', 'severity_by_year.png')

In [ ]:
by_source = pd.crosstab(df['Source'], df[config.TARGET])
by_source.assign(total=by_source.sum(axis=1))

In [ ]:
plot_severity_share(by_source, 'Severity mix by Source', 'severity_by_source.png')

## Checking the "constant" columns

`clean()` drops `Country` and `Turning_Loop` because they should only have one value.

In [ ]:
for col in ['Country', 'Turning_Loop']:
    print(df[col].value_counts(dropna=False), '\n')

## Observations

- Missing values: End_Lat/End_Lng are missing in 44% of rows.
    Precipitation is missing in 28.5%, Wind_Chill 25.9%, Wind_Speed 7.4%, and the other weather
    columns about 2%. Everything else is basically complete.
- Severity overall: very imbalanced. Severity 2 is about 80% of rows, 3 is 17%, 4 is 2.7%
    and 1 is under 1%. A model that always guesses 2 gets ~79% accuracy, so we use macro-F1.
- By year: the severity mix changes a lot. Severity 3 drops from about a third of accidents
    in 2016-2018 to 0% in 2023, and severity 1 only really shows up in 2020 and 2022. That looks
    like a change in how severity was labelled, not in real accidents.
- By Source: Source1 is 91% severity 2, while Source2 and Source3 are about 65% severity 2
    and a third severity 3. So the label also depends on which API reported the accident.
- What this means for cleaning and modelling: dropping Country and Turning_Loop is fine since both have only one value.
    Precipitation gets filled with 0 plus a missing flag, and the other weather NaNs are left for
    the model pipelines to impute on train only. The year/Source labelling shift is a limitation
    for the report, and the team should decide whether `year` or `Source` goes in as a feature.